#  Custom Traffic Violation Detector 

###  Pipeline Overview
This notebook trains a custom **YOLOv8-Nano** object detector from scratch for 100 continuous epochs. It locates and classifies three distinct classes from multiple blended datasets:
1. **`Class 0: helmet`**
2. **`Class 1: no_helmet`**
3. **`Class 2: license_plate`**

###  Key Architectural Optimizations Built-In:
- **Asynchronous Multi-Threaded Sync**: Pushes `best.pt` to your Hugging Face Model Hub in a background thread so training never stops/blocks during uploads.
- **Safe-Mode Dataloading**: Configured with `batch=64` and `workers=2` to guarantee zero shared-memory (`/dev/shm`) or system OOM crashes on Hugging Face Spaces.
- **Robust Data Augmentations**: Includes custom Mosaic (1.0), Mixup (0.1), and affine/color transforms to completely prevent overfitting.
- **Live Visual Monitoring**: Automatically updates a beautiful `live_report_plots.png` file on disk after every single epoch. You can view your report graphs in real-time while training is running!

##  Cell 1: Verify GPU Environment

In [ ]:
!nvidia-smi

##  Cell 2: Configure Credentials & Space Secrets

In [ ]:
import os, json
from pathlib import Path

# Securely read secrets from Space Environment Settings
ROBOFLOW_KEY    = os.environ.get('ROBOFLOW_KEY', '')
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
KAGGLE_KEY      = os.environ.get('KAGGLE_KEY', '')
HF_TOKEN        = os.environ.get('HF_TOKEN', '')

# ── SET YOUR HF USERNAME ───────────────────────────────────────
HF_USERNAME     = 'Advaya-ab'
# ────────────────────────────────────────────────────────────────

assert ROBOFLOW_KEY,    ' Please set ROBOFLOW_KEY in Space Secrets!'
assert KAGGLE_USERNAME, ' Please set KAGGLE_USERNAME in Space Secrets!'
assert KAGGLE_KEY,      ' Please set KAGGLE_KEY in Space Secrets!'
assert HF_TOKEN,        ' Please set HF_TOKEN in Space Secrets!'

# Expose token to huggingface_hub client
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print(' Credentials successfully loaded!')

##  Cell 3: Install Dependencies & Setup Headless OpenCV

In [ ]:
# Install packages
!pip install -q ultralytics roboflow kaggle huggingface_hub pyyaml pandas matplotlib

# Clean up standard OpenCV to avoid libGL.so dependency crashes inside Docker containers
!pip uninstall -y opencv-python opencv-python-headless
!pip install opencv-python-headless --force-reinstall

import ultralytics
ultralytics.checks()

# Write Kaggle Auth Configuration File
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print(' OpenCV and Kaggle environment successfully configured!')

## Cell 4: Download Datasets

In [ ]:
print('Downloading all custom datasets from Kaggle & Roboflow...')

# 1. Download Kaggle datasets
!kaggle datasets download -d aneesarom/rider-with-helmet-without-helmet-number-plate -p ./raw/aneesarom --unzip -q
!kaggle datasets download -d ronakgohil/license-plate-dataset -p ./raw/ronakgohil --unzip -q

# 2. Download Roboflow datasets
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_KEY)
rf.workspace('onlytusik').project('helmet-detection-fk6is').version(1).download('yolov8', location='./raw/onlytusik')
rf.workspace('imagerecognition-43zpb').project('helmet-detection-ntbfz').version(6).download('yolov8', location='./raw/imagerecognition')
rf.workspace('roboflow-universe-projects').project('license-plate-recognition-rxg4e').version(4).download('yolov8', location='./raw/roboflow_lp')

print(' All raw datasets successfully downloaded to ./raw/!')

##  Cell 5: Remap Annotations & Merge Classes

In [ ]:
import os, shutil, yaml, random
from pathlib import Path

datasets = {
    'aneesarom':       Path('./raw/aneesarom'),
    'ronakgohil':      Path('./raw/ronakgohil'),
    'onlytusik':       Path('./raw/onlytusik'),
    'imagerecognition':Path('./raw/imagerecognition'),
    'roboflow_lp':     Path('./raw/roboflow_lp'),
}

# ── Define class mappings (Mapping diverse labels to: 0=helmet, 1=no_helmet, 2=license_plate) ──
CLASS_MAPS = {
    'aneesarom': {
        0: 0,     # 'with helmet'      → helmet (0)
        1: 1,     # 'without helmet'   → no_helmet (1)
        2: None,  # 'rider'            → drop (None)
        3: 2,     # 'number plate'     → license_plate (2)
    },
    'onlytusik': {
        0: 0,     # 'with helmet'      → helmet (0)
        1: 1,     # 'without helmet'   → no_helmet (1)
    },
    'imagerecognition': {
        0: None,  # 'Bicyclist'        → drop (None)
        1: None,  # 'driver'           → drop (None)
        2: 0,     # 'helmet'           → helmet (0)
        3: 1,     # 'no-helmet'        → no_helmet (1)
    },
    'roboflow_lp': {
        0: 2,     # 'License_Plate'    → license_plate (2)
    },
    'ronakgohil': {
        0: 2,     # 'license_plate'    → license_plate (2)
    },
}

def remap_label(src, dst, cmap):
    dst.parent.mkdir(parents=True, exist_ok=True)
    lines_out = []
    if src.exists():
        for line in src.read_text().splitlines():
            if not line.strip(): continue
            parts = line.split()
            tgt = cmap.get(int(parts[0]), None)
            if tgt is not None:
                parts[0] = str(tgt)
                lines_out.append(' '.join(parts))
    dst.write_text('\n'.join(lines_out))

pool_imgs = Path('./pool/images')
pool_lbls = Path('./pool/labels')
pool_imgs.mkdir(parents=True, exist_ok=True)
pool_lbls.mkdir(parents=True, exist_ok=True)

for ds_name, ds_path in datasets.items():
    cmap = CLASS_MAPS.get(ds_name, {})
    for split in ['train', 'val', 'test']:
        for base in [ds_path / 'images' / split, ds_path / split / 'images']:
            if not base.exists(): continue
            lbl_base = base.parent.parent / 'labels' / split
            if not lbl_base.exists():
                lbl_base = base.parent / 'labels'
            for img in base.glob('*.*'):
                if img.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}: continue
                new_stem = f'{ds_name}_{img.stem}'
                shutil.copy2(img, pool_imgs / (new_stem + img.suffix))
                remap_label(
                    lbl_base / (img.stem + '.txt'),
                    pool_lbls / (new_stem + '.txt'),
                    cmap
                )
    print(f'   {ds_name} remapped and merged')

print(f'\nTotal combined images in raw pool: {len(list(pool_imgs.glob("*.*")))}')

## Cell 6: Train/Val Split & data.yaml Generation

In [ ]:
import random, shutil, yaml
from collections import Counter

out = Path('./merged_dataset')
for s in ['train','val']:
    (out/'images'/s).mkdir(parents=True, exist_ok=True)
    (out/'labels'/s).mkdir(parents=True, exist_ok=True)

all_imgs = sorted(pool_imgs.glob('*.*'))
random.seed(42)
random.shuffle(all_imgs)

n_val = int(len(all_imgs) * 0.15) # 15% Validation Split
val_set = set(img.name for img in all_imgs[:n_val])

for img in all_imgs:
    split = 'val' if img.name in val_set else 'train'
    shutil.copy2(img, out/'images'/split/img.name)
    lbl = pool_lbls / (img.stem + '.txt')
    if lbl.exists():
        shutil.copy2(lbl, out/'labels'/split/lbl.name)

# Generate YOLOv8 config file (data.yaml)
with open(out/'data.yaml', 'w') as f:
    yaml.dump({
        'path': str(out.resolve()),
        'train': 'images/train',
        'val':   'images/val',
        'nc': 3,
        'names': {0:'helmet', 1:'no_helmet', 2:'license_plate'}
    }, f, default_flow_style=False)

n_train = len(all_imgs) - n_val
print(f'Train size: {n_train} | Val size: {n_val} | data.yaml successfully written')

# Class distribution verification check
counts = Counter()
for lbl in (out/'labels'/'train').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        if line.strip(): counts[int(line.split()[0])] += 1

names = {0:'helmet', 1:'no_helmet', 2:'license_plate'}
print('\nClass distribution (bounding box counts in training set):')
for c, n in sorted(counts.items()):
    print(f'  {names[c]:<20}: {n}')

##  Cell 7: Train Continuous 100 Epochs (From Scratch)

Runs a full training pipeline using safe multiprocessing parameters. It registers an **Asynchronous Multi-Threaded Sync Callback** to back up checkpoints dynamically. Upon completion, it automatically uploads the authentic 100-epoch `results.png` learning curves directly to Hugging Face.

###  Live Monitor Feature:
While this cell is running, it generates a clean **`live_report_plots.png`** inside the training directory (e.g., `./runs/detect/model2_final/live_report_plots.png`) **after every single epoch**! You can double-click this image in Jupyter to watch your charts build in real-time.

In [ ]:
from ultralytics import YOLO
from huggingface_hub import HfApi
from pathlib import Path
import threading
import pandas as pd
import matplotlib.pyplot as plt

# ── Asynchronous Hugging Face Sync Callback ───────────────────
def _do_upload(last_path, best_path, repo_id, token, epoch):
    """Background worker thread to upload weights without pausing training"""
    try:
        api = HfApi(token=token)
        api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)
        
        # 1. ALWAYS upload the absolute latest checkpoint ("last.pt")
        if Path(last_path).exists():
            api.upload_file(
                path_or_fileobj=str(last_path),
                path_in_repo='last.pt',
                repo_id=repo_id,
                repo_type='model'
            )
        # 2. ALWAYS upload the absolute best checkpoint ("best.pt")
        if Path(best_path).exists():
            api.upload_file(
                path_or_fileobj=str(best_path),
                path_in_repo='model2_helmet_plate.pt',
                repo_id=repo_id,
                repo_type='model'
            )
        print(f'\n[HF Hub Sync] Epoch {epoch}: Checkpoints successfully uploaded in background! 💾')
    except Exception as e:
        print(f'\n[HF Hub Sync Error] Epoch {epoch} upload failed: {e}')

# ── Real-Time Live Plotting Callback ─────────────────────────
def live_plot_callback(trainer):
    """Generates and updates a beautiful live learning curve plot after every epoch"""
    csv_path = Path(trainer.save_dir) / 'results.csv'
    if not csv_path.exists():
        return
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        if len(df) == 0: return
        
        epochs = df['epoch']
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Left Plot: Live Loss
        if 'train/box_loss' in df.columns:
            ax1.plot(epochs, df['train/box_loss'], label='Train Box Loss', color='#ef4444', linewidth=2)
            ax1.plot(epochs, df['val/box_loss'], label='Val Box Loss', color='#f97316', linestyle='--', linewidth=2)
            ax1.plot(epochs, df['train/cls_loss'], label='Train Cls Loss', color='#3b82f6', linewidth=2)
            ax1.plot(epochs, df['val/cls_loss'], label='Val Cls Loss', color='#06b6d4', linestyle='--', linewidth=2)
        ax1.set_title(f'Live Loss (Epoch {trainer.epoch + 1})', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss Value')
        ax1.grid(True, linestyle='--', alpha=0.5)
        ax1.legend()
        
        # Right Plot: Live Accuracy
        if 'metrics/mAP50(B)' in df.columns:
            ax2.plot(epochs, df['metrics/mAP50(B)'], label='mAP @ 0.50', color='#10b981', linewidth=2.5)
            ax2.plot(epochs, df['metrics/mAP50-95(B)'], label='mAP @ 0.50:0.95', color='#059669', linewidth=2.5)
        ax2.set_title(f'Live Validation Accuracy (mAP)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy Score (0 to 1)')
        ax2.set_ylim(0, 1.0)
        ax2.grid(True, linestyle='--', alpha=0.5)
        ax2.legend()
        
        plt.tight_layout()
        plt.savefig(Path(trainer.save_dir) / 'live_report_plots.png', dpi=150)
        plt.close()
    except Exception:
        pass

def on_fit_epoch_end_callback(trainer):
    # 1. Trigger live curves plotting
    live_plot_callback(trainer)
    # 2. Trigger asynchronous online checkpoint syncing
    repo_id = f'{HF_USERNAME}/traffic-violation-model2'
    t = threading.Thread(
        target=_do_upload,
        args=(trainer.last, trainer.best, repo_id, HF_TOKEN, trainer.epoch + 1),
        daemon=True
    )
    t.start()

# ── Load model ────────────────────────────────────────────────
model = YOLO('yolov8n.pt')

# ── Register the Callback ─────────────────────────────────────
model.add_callback('on_fit_epoch_end', on_fit_epoch_end_callback)

# ── Start Training ────────────────────────────────────────────
results = model.train(
    data      = './merged_dataset/data.yaml',
    epochs    = 100,       # 100 total epochs
    imgsz     = 640,       # standard size
    batch     = 64,        # safe OOM-proof batch size
    workers   = 2,         # safe SHM limits for Hugging Face Docker
    device    = 0,         # A10G Small GPU
    name      = 'model2_final',
    patience  = 25,        # Early stopping patience
    save      = True, 

    # Advanced Anti-Overfitting Augmentations
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    degrees   = 5.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,       # Mosaic augmentation (combines 4 images)
    mixup     = 0.1,       # Mixup augmentation
    cls       = 1.5,
    plots     = True,
)

# ── Post-Training Logs Upload ─────────────────────────────────
try:
    print(' Uploading final 100-epoch authentic plots and CSV logs to Hugging Face...')
    api = HfApi(token=HF_TOKEN)
    repo_id = f'{HF_USERNAME}/traffic-violation-model2'
    
    # Upload the results PNG chart
    api.upload_file(
        path_or_fileobj=str(Path(results.save_dir) / 'results.png'),
        path_in_repo='results.png',
        repo_id=repo_id, 
        repo_type='model'
    )
    # Upload the results CSV logs
    api.upload_file(
        path_or_fileobj=str(Path(results.save_dir) / 'results.csv'),
        path_in_repo='results.csv',
        repo_id=repo_id, 
        repo_type='model'
    )
    # Upload confusion matrix
    api.upload_file(
        path_or_fileobj=str(Path(results.save_dir) / 'confusion_matrix.png'),
        path_in_repo='confusion_matrix.png',
        repo_id=repo_id, 
        repo_type='model'
    )
    print(' All authentic training logs and plots successfully backed up!')
except Exception as e:
    print(f'Failed to upload plots/logs to HF: {e}')

print(' Continuous 100-Epoch Training pipeline fully complete!')

## Cell 8: Model Evaluation & Customized Report Plots

Validates the absolute best saved model (`best.pt`), computes final project accuracy, and generates **highly polished, custom, self-explanatory graphs** tailored for your project report.

In [ ]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
from huggingface_hub import HfApi

# 1. Load the BEST saved weights for final evaluation
best_weights = './runs/detect/model2_final/weights/best.pt'
model_best = YOLO(best_weights)

# 2. Run Validation on validation split
metrics = model_best.val(data='./merged_dataset/data.yaml', imgsz=640, device=0)

print(f'\n FINAL METRICS REPORT:')
print(f'  • Overall mAP50    : {metrics.box.map50:.4f}')
print(f'  • Overall mAP50-95 : {metrics.box.map:.4f}')
for i, name in enumerate(['helmet', 'no_helmet', 'license_plate']):
    print(f'    - {name:<16} mAP50: {metrics.box.maps[i]:.4f}')

# 3. Plot Highly Self-Explanatory, Clean Report Charts
csv_path = Path(best_weights).parent.parent / 'results.csv'
if csv_path.exists():
    print('\n Generating Custom Self-Explanatory Report Plots...')
    df = pd.read_csv(csv_path)
    # Remove trailing whitespaces in YOLOv8 CSV column headers
    df.columns = df.columns.str.strip()
    
    epochs = df['epoch']
    
    plt.figure(figsize=(15, 6))
    
    # Chart A: Training vs Validation Losses
    plt.subplot(1, 2, 1)
    plt.plot(epochs, df['train/box_loss'], label='Train Box Loss', color='#ef4444', linewidth=2)
    plt.plot(epochs, df['val/box_loss'], label='Val Box Loss', color='#f97316', linestyle='--', linewidth=2)
    plt.plot(epochs, df['train/cls_loss'], label='Train Cls Loss', color='#3b82f6', linewidth=2)
    plt.plot(epochs, df['val/cls_loss'], label='Val Cls Loss', color='#06b6d4', linestyle='--', linewidth=2)
    plt.title('Training & Validation Convergence (Loss)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss Value')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    
    # Chart B: Accuracy Progression (mAP50 and mAP50-95)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, df['metrics/mAP50(B)'], label='mAP @ 0.50 (Overlap)', color='#10b981', linewidth=2.5)
    plt.plot(epochs, df['metrics/mAP50-95(B)'], label='mAP @ 0.50:0.95 (Strict)', color='#059669', linewidth=2.5)
    plt.title('Validation Accuracy Progress (mAP)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy Score (0 to 1)')
    plt.ylim(0, 1.0)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    
    plt.tight_layout()
    custom_plot_path = Path(best_weights).parent.parent / 'custom_report_plots.png'
    plt.savefig(custom_plot_path, dpi=300)
    plt.show()
    
    # Upload the custom clean plot to Hugging Face
    try:
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=str(custom_plot_path),
            path_in_repo='custom_report_plots.png',
            repo_id=f'{HF_USERNAME}/traffic-violation-model2',
            repo_type='model'
        )
        print(' Custom self-explanatory plots successfully saved and uploaded!')
    except Exception as e:
        print(f'Failed to upload custom plot to HF: {e}')

##  Cell 9: Run Visual Bounding Box Tests

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import random
from IPython.display import Image, display

# Load best weights
best_weights = './runs/detect/model2_final/weights/best.pt'
model_best = YOLO(best_weights)

# Grab random images from the validation set
val_imgs = list(Path('./merged_dataset/images/val').glob('*.*'))
test_imgs = random.sample(val_imgs, min(3, len(val_imgs)))

# Run inference
results = model_best.predict(test_imgs, conf=0.4, imgsz=640, save=True)

# Show prediction outputs
print('\nShowing sample test predictions:')
for r in results:
    display(Image(filename=str(Path(r.save_dir) / Path(r.path).name)))